# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.496500,0.527291,0.637203,0.806670,...,0.3,0.815075,0.788450,0.867595,0.911292,0.783514,0.868510,0.784140,0.618609,0.543036
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.340099,0.303278,0.512236,0.734278,...,0.7,0.686883,0.686778,0.594025,0.698730,0.695484,0.772514,0.666212,0.572044,0.446302
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.127990,0.192701,0.330117,0.533609,...,0.5,0.580053,0.607686,0.523752,0.600009,0.614942,0.717012,0.570198,0.631260,0.440385
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.249994,-1.823374,-0.834575,0.001354,...,0.0,0.002635,0.002627,0.013206,0.025528,0.000000,0.005494,0.063219,0.000000,0.076485
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.090382,-0.011299,0.200067,0.386963,...,0.3,0.397511,0.457883,0.364956,0.265409,0.452112,0.482416,0.418672,0.245866,0.244423


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,-1.721325,-0.872993,0.005473,...,0.0,0.006534,0.003712,0.009642,0.010313,0.000000,0.019849,0.061327,0.000000,0.076454
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,-0.919863,-0.289991,0.054146,...,0.1,0.063555,0.110500,0.044563,0.145672,0.047794,0.012173,0.144461,0.102869,0.089535
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,-1.809246,-0.917667,0.003687,...,0.0,0.004404,0.003123,0.008417,0.006856,0.000000,0.010304,0.057929,0.000000,0.076454
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.328363,0.387739,0.676445,...,0.3,0.670446,0.712745,0.648495,0.504756,0.637237,0.444558,0.628242,0.183451,0.215908
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.644014,0.670147,0.852891,...,0.8,0.835307,0.855796,0.796202,0.823601,0.787922,0.939648,0.787450,0.823541,0.643512


# Machine Learning

In [18]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=400, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-25 16:34:16,350] A new study created in memory with name: no-name-95dac7a0-9615-42f6-b3a5-62c49ab455a7
Best trial: 0. Best value: 0.94963:   0%|                                           | 1/400 [00:22<2:30:34, 22.64s/it]

[I 2026-05-25 16:34:38,980] Trial 0 finished with value: 0.9496297555027505 and parameters: {'n_estimators': 55, 'learning_rate': 0.02143359339870089, 'max_depth': 4, 'num_leaves': 16, 'min_child_samples': 171, 'subsample': 0.9496233692684151, 'colsample_bytree': 0.8664141178120989, 'reg_alpha': 0.07954603021890694, 'reg_lambda': 10.82244197290681}. Best is trial 0 with value: 0.9496297555027505.


Best trial: 9. Best value: 0.949697:   0%|▏                                         | 2/400 [00:28<1:25:18, 12.86s/it]

[I 2026-05-25 16:34:44,987] Trial 9 finished with value: 0.9496967419959775 and parameters: {'n_estimators': 200, 'learning_rate': 0.03948082767141426, 'max_depth': 3, 'num_leaves': 2, 'min_child_samples': 144, 'subsample': 0.7328384884066917, 'colsample_bytree': 0.6371658666252459, 'reg_alpha': 0.031874671900745104, 'reg_lambda': 15.5960535682379}. Best is trial 9 with value: 0.9496967419959775.


Best trial: 9. Best value: 0.949697:   1%|▎                                           | 3/400 [00:30<52:48,  7.98s/it]

...

[I 2026-05-25 17:14:48,172] Trial 397 finished with value: 0.9501126895435823 and parameters: {'n_estimators': 285, 'learning_rate': 0.045967090239230936, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 113, 'subsample': 0.8386226627975651, 'colsample_bytree': 0.6293098417118342, 'reg_alpha': 0.0006096308730616138, 'reg_lambda': 8.754968483628737}. Best is trial 287 with value: 0.950124856169478.


Best trial: 287. Best value: 0.950125: 100%|███████████████████████████████████████▉| 399/400 [40:32<00:02,  2.55s/it]

[I 2026-05-25 17:14:48,664] Trial 399 finished with value: 0.9500993756367541 and parameters: {'n_estimators': 283, 'learning_rate': 0.04605759095727158, 'max_depth': 4, 'num_leaves': 6, 'min_child_samples': 118, 'subsample': 0.8359544108751806, 'colsample_bytree': 0.6005173377489544, 'reg_alpha': 0.01590379040231663, 'reg_lambda': 9.2364013825848}. Best is trial 287 with value: 0.950124856169478.


Best trial: 287. Best value: 0.950125: 100%|████████████████████████████████████████| 400/400 [40:32<00:00,  6.08s/it]

[I 2026-05-25 17:14:48,986] Trial 396 finished with value: 0.9501110300459077 and parameters: {'n_estimators': 285, 'learning_rate': 0.045538941600613905, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 133, 'subsample': 0.8362400214685676, 'colsample_bytree': 0.8304872008647101, 'reg_alpha': 26.29541388066824, 'reg_lambda': 9.2125971146953}. Best is trial 287 with value: 0.950124856169478.
Best trial score:
0.950124856169478

Best params:
{'n_estimators': 297, 'learning_rate': 0.0445850002648349, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 142, 'subsample': 0.8266384653021891, 'colsample_bytree': 0.6572010325519047, 'reg_alpha': 9.038880464820691, 'reg_lambda': 1.5268153771055861}


In [30]:
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=200, n_jobs=-1, show_progress_bar=True)

Best trial: 287. Best value: 0.950125:   0%|▏                                         | 1/200 [00:17<58:55, 17.77s/it]

[I 2026-05-25 17:16:35,812] Trial 404 pruned. 


Best trial: 287. Best value: 0.950125:   1%|▍                                         | 2/200 [00:30<48:00, 14.55s/it]

[I 2026-05-25 17:16:48,129] Trial 409 finished with value: 0.9498337629231106 and parameters: {'n_estimators': 80, 'learning_rate': 0.04012169840323628, 'max_depth': 4, 'num_leaves': 11, 'min_child_samples': 133, 'subsample': 0.8893044550869753, 'colsample_bytree': 0.6101449651333914, 'reg_alpha': 11.34043628122149, 'reg_lambda': 83.88357159692966}. Best is trial 287 with value: 0.950124856169478.


Best trial: 287. Best value: 0.950125:   2%|▌                                       | 3/200 [01:17<1:36:57, 29.53s/it]

...

[I 2026-05-25 17:37:10,757] Trial 596 finished with value: 0.9501254766249131 and parameters: {'n_estimators': 278, 'learning_rate': 0.046555514464615984, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 182, 'subsample': 0.9432191564927672, 'colsample_bytree': 0.697525468355304, 'reg_alpha': 21.410615882582412, 'reg_lambda': 0.0009087781547661486}. Best is trial 535 with value: 0.950131034246073.


Best trial: 535. Best value: 0.950131:  99%|███████████████████████████████████████▌| 198/200 [20:53<00:05,  2.93s/it]

[I 2026-05-25 17:37:11,857] Trial 598 finished with value: 0.9501182388857213 and parameters: {'n_estimators': 274, 'learning_rate': 0.04760248011313686, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 189, 'subsample': 0.9437070288659243, 'colsample_bytree': 0.6952017146238905, 'reg_alpha': 19.8729414903485, 'reg_lambda': 30.906654287034915}. Best is trial 535 with value: 0.950131034246073.


Best trial: 535. Best value: 0.950131: 100%|████████████████████████████████████████| 200/200 [20:54<00:00,  6.27s/it]

[I 2026-05-25 17:37:12,824] Trial 597 finished with value: 0.9501147387330148 and parameters: {'n_estimators': 275, 'learning_rate': 0.047309941803596725, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 166, 'subsample': 0.9657079572527418, 'colsample_bytree': 0.6959679310924869, 'reg_alpha': 18.501810904114123, 'reg_lambda': 0.00011024203601341157}. Best is trial 535 with value: 0.950131034246073.
[I 2026-05-25 17:37:12,962] Trial 599 finished with value: 0.950125978671663 and parameters: {'n_estimators': 276, 'learning_rate': 0.04707590311762272, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 188, 'subsample': 0.9427697203272083, 'colsample_bytree': 0.7027180140408709, 'reg_alpha': 18.8598099478477, 'reg_lambda': 17.42449716243865}. Best is trial 535 with value: 0.950131034246073.


In [31]:
stacking_lgbm = LGBMClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013048 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5621
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668


# Submission

In [32]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [33]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [34]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [35]:
X_train.columns

Index(['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba',
       'pca_knn_proba', 'ridge_logit', 'truncatedsvd_linear_svc_logit',
       'truncatedsvd_ridge_logit', 'sgdclassifier_proba',
       'truncatedsvd_lgbm_proba', 'truncatedsvd_knn_proba',
       'truncatedsvd_nystroem_knn_proba', 'truncatedsvd_rbfsampler_knn_proba',
       'truncatedsvd_logistic_regression_proba',
       'truncatedsvd_nystroem_logistic_regression_proba',
       'truncatedsvd_nystroem_sgdclassifier_proba',
       'truncatedsvd_rbfsampler_logistic_regression_proba',
       'truncatedsvd_rbfsampler_sgdclassifier_proba',
       'voting_lgbm_and_cat_and_xgb_and_hist_proba',
       'voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier',
       'voting_truncatedsvd_nytroem_knn_and_